In [1]:
import pandas as pd

df = pd.read_csv('../creditcard.csv')

In [2]:
import sqlite3

conn = sqlite3.connect('../fraud_detection.db')
cursor = conn.cursor()

df_sql = df.copy()
df_sql.columns = df_sql.columns.str.lower()
df_sql = df_sql.rename(columns={
    'time': 'time_seconds',
    'class': 'is_fraud'
})
df_sql.insert(0, 'transaction_id', range(1, len(df_sql)+1))

df_sql.to_sql('transactions', conn, 
              if_exists='replace', 
              index=False)

print("Database created successfully!")
print("Total records loaded:", len(df_sql))

Database created successfully!
Total records loaded: 284807


In [4]:
print("QUERY 1 - FRAUD OVERVIEW")
print("-" * 40)
query1 = """
SELECT 
    is_fraud,
    COUNT(*) as total_transactions,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(MAX(amount), 2) as max_amount,
    ROUND(MIN(amount), 2) as min_amount
FROM transactions
GROUP BY is_fraud
"""
result1 = pd.read_sql_query(query1, conn)
print(result1)

QUERY 1 - FRAUD OVERVIEW
----------------------------------------
   is_fraud  total_transactions  avg_amount  max_amount  min_amount
0         0              284315       88.29    25691.16         0.0
1         1                 492      122.21     2125.87         0.0


In [5]:
print("QUERY 2 - PEAK FRAUD HOURS (FIXED)")
print("-" * 40)
query2_fixed = """
SELECT 
    CAST(time_seconds / 3600 AS INTEGER) as hour,
    COUNT(*) as total_transactions,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) as fraud_count,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as fraud_rate
FROM transactions
GROUP BY hour
ORDER BY fraud_rate DESC
LIMIT 10
"""
result2_fixed = pd.read_sql_query(query2_fixed, conn)
print(result2_fixed)

QUERY 2 - PEAK FRAUD HOURS (FIXED)
----------------------------------------
   hour  total_transactions  fraud_count  fraud_rate
0    26                1752           36        2.05
1    28                1127           17        1.51
2     2                1576           21        1.33
3     3                1821           13        0.71
4     7                3368           23        0.68
5     5                1681           11        0.65
6     4                1082            6        0.55
7    11                8517           43        0.50
8    25                2003            8        0.40
9    23                6082           17        0.28


In [6]:
print("QUERY 3 - FRAUD BY AMOUNT RANGE")
print("-" * 40)
query3 = """
SELECT 
    CASE 
        WHEN amount < 10 THEN '1. Below 10'
        WHEN amount < 50 THEN '2. 10 to 50'
        WHEN amount < 100 THEN '3. 50 to 100'
        WHEN amount < 500 THEN '4. 100 to 500'
        ELSE '5. Above 500'
    END as amount_range,
    COUNT(*) as total_transactions,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) as fraud_count,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as fraud_rate
FROM transactions
GROUP BY amount_range
ORDER BY amount_range
"""
result3 = pd.read_sql_query(query3, conn)
print(result3)

QUERY 3 - FRAUD BY AMOUNT RANGE
----------------------------------------
    amount_range  total_transactions  fraud_count  fraud_rate
0    1. Below 10               97314          249        0.26
1    2. 10 to 50               92390           56        0.06
2   3. 50 to 100               37718           57        0.15
3  4. 100 to 500               47893           95        0.20
4   5. Above 500                9492           35        0.37
